In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

> **Note:** Funkcje Qiskit to eksperymentalna funkcjonalność dostępna wyłącznie dla użytkowników planów IBM Quantum&reg; Premium Plan, Flex Plan oraz On-Prem (za pośrednictwem IBM Quantum Platform API). Mają status wersji podglądowej i mogą ulec zmianie.

## Omówienie
W chemii kwantowej problem struktury elektronowej polega na znalezieniu rozwiązań elektronowego równania Schrödingera — kwantowych funkcji falowych opisujących zachowanie elektronów w danym układzie. Funkcje te są wektorami amplitud zespolonych, przy czym każda amplituda odpowiada wkładowi jednej z możliwych konfiguracji elektronowych.

Stan podstawowy to funkcja falowa układu o najniższej energii i odgrywa szczególną rolę w badaniu układów molekularnych. Najbardziej dokładne podejście do obliczania stanu podstawowego uwzględnia wszystkie możliwe konfiguracje elektronowe, lecz dla większych układów staje się niewykonalne, ponieważ liczba konfiguracji rośnie wykładniczo wraz z rozmiarem układu.

Handover Iterative Variational Quantum Eigensolver (HI-VQE) to nowatorska hybrydowa metoda kwantowo-klasyczna służąca do precyzyjnego szacowania stanu podstawowego układów molekularnych. Łączy sprzęt kwantowy z obliczeniami klasycznymi — procesory kwantowe służą do efektywnego przeszukiwania kandydatów na konfiguracje elektronowe, a wynikowa funkcja falowa jest obliczana na komputerach klasycznych. Generując zwarte, lecz chemicznie dokładne funkcje falowe, HI-VQE wspomaga badania i odkrycia w chemii kwantowej oraz nauce o materiałach.

![Obraz przedstawiający ogólny zarys algorytmu HI-VQE firmy Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE zmniejsza złożoność obliczeniową problemu struktury elektronowej, efektywnie szacując stan podstawowy z dużą dokładnością. Skupia się na starannie dobranym podzbiorze najważniejszych konfiguracji elektronowych, optymalizując jednocześnie dokładność i wydajność.

Łącząc zalety komputerów klasycznych i kwantowych, HI-VQE iteracyjnie udoskonala bieżące przybliżenie funkcji falowej. Unikalne techniki budowy podprzestrzeni pomagają zwiększyć efektywność doboru konfiguracji, zapewniając użytkownikom większą kontrolę obliczeniową i lepszą dokładność symulacji chemii kwantowej.

Jeśli chcesz dokładniej poznać algorytm, możesz [przeczytać powiązaną publikację naukową](https://arxiv.org/abs/2503.06292).

## Opis
Liczba konfiguracji elektronowych w układzie molekularnym rośnie wykładniczo wraz z rozmiarem układu. Jednak dla pewnych stanów elektronowych, takich jak stan podstawowy, zazwyczaj tylko niewielki ułamek konfiguracji wnosi istotny wkład do energii stanu. Metody wybranej interakcji konfiguracyjnej (SCI) wykorzystują tę rzadkość do obniżenia kosztów obliczeniowych, identyfikując i koncentrując się na najbardziej istotnych konfiguracjach. Podzbiór tych konfiguracji określa się mianem podprzestrzeni.

HI-VQE wykorzystuje wrodzoną efektywność komputerów kwantowych w reprezentowaniu układów molekularnych, wspomagając przeszukiwanie podprzestrzeni. Integruje klasyczne i kwantowe procedury, aby rozwiązać problem struktury elektronowej z dużą dokładnością. W odróżnieniu od istniejących kwantowych metod SCI, HI-VQE łączy trenowanie wariacjne, iteracyjne budowanie podprzestrzeni oraz wstępne przesiewanie konfiguracji metodą pre-diagonalizacji, co zwiększa efektywność przez ograniczenie liczby pomiarów kwantowych, iteracji i kosztów klasycznej diagonalizacji. HI-VQE można zatem stosować do większych układów molekularnych wymagających większej liczby Qubitów, a koszt rozwiązania problemu danego rozmiaru z tym samym stopniem dokładności ulega zmniejszeniu.

![Obraz przedstawiający szczegółowy opis każdego kroku algorytmu HI-VQE firmy Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

Aby obliczyć stan podstawowy układu, HI-VQE najpierw używa klasycznego pakietu chemicznego PySCF do wygenerowania reprezentacji molekularnej na podstawie danych wejściowych podanych przez użytkownika, takich jak geometria molekularna i inne informacje o cząsteczce. Następnie wchodzi w hybrydową pętlę optymalizacji kwantowo-klasycznej, iteracyjnie udoskonalając podprzestrzeń, by optymalnie reprezentować stan podstawowy przy minimalizacji liczby zawartych konfiguracji. Pętla działa do momentu spełnienia kryteriów zbieżności, takich jak rozmiar podprzestrzeni lub stabilność energii; po ich osiągnięciu wynikiem jest obliczona funkcja falowa stanu podstawowego wraz z energią. Wyniki te można wykorzystać do konstruowania dokładnych powierzchni energii potencjalnej i przeprowadzania dalszej analizy układu.

Pętla optymalizacji skupia się na dostosowywaniu parametrów Circuit kwantowego w celu wygenerowania wysokiej jakości podprzestrzeni. HI-VQE oferuje trzy opcje Circuit kwantowego: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) oraz [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). Optymalizacja jest inicjalizowana blisko stanu referencyjnego Hartree-Focka ze względu na jego ogólną przydatność. Następnie Circuit jest wykonywany na urządzeniu kwantowym i z wynikowego stanu kwantowego pobierane są próbki konfiguracji, które są zwracane jako ciągi binarne. Ze względu na szumy urządzenia kwantowego niektóre próbkowane konfiguracje mogą być fizycznie nieprawidłowe — nie zachowują liczby elektronów ani spinu. HI-VQE rozwiązuje ten problem, stosując proces odtwarzania konfiguracji z pakietu [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), dzięki czemu użytkownicy mogą albo poprawiać nieprawidłowe konfiguracje, albo je odrzucać.

Prawidłowe konfiguracje przechodzą opcjonalny krok przesiewania, usuwający te, które przewiduje się jako wnoszące minimalny wkład. Zmniejsza to wymiar podprzestrzeni, obniżając tym samym koszt kroku diagonalizacji. Jeśli przesiewanie jest włączone, na podstawie prawidłowych konfiguracji konstruowany jest wstępny hamiltonian podprzestrzeni, a diagonalizacja wykonywana jest z bardzo luźnymi kryteriami zakończenia. Choć dokładność wynikowych amplitud dla każdej konfiguracji jest niska, jest ona skuteczna do przewidywania, które konfiguracje pominąć w podprzestrzeni w danej iteracji, i jest szybka do obliczenia.

Wybrane konfiguracje są dodawane do podprzestrzeni, a hamiltonian układu jest rzutowany do tej podprzestrzeni. Podprzestrzeń aktualizuje się iteracyjnie, zachowując najważniejsze konfiguracje pomiędzy iteracjami. Podejście to różni się od metod alternatywnych tym, że Circuit kwantowy nie musi przybliżać pełnego stanu podstawowego w każdym kroku.

Następnie hamiltonian podprzestrzeni jest klasycznie diagonalizowany w celu uzyskania najniższej wartości własnej i odpowiadającego jej wektora własnego, reprezentującego przybliżenie stanu podstawowego i jego energii. W miarę jak jakość podprzestrzeni poprawia się w kolejnych iteracjach, obliczony stan podstawowy coraz lepiej przybliża rzeczywisty stan podstawowy. Na tym etapie można przeprowadzić dodatkowy krok przesiewania, usuwający z podprzestrzeni wszelkie konfiguracje, które nie wnoszą istotnego wkładu do obliczonego stanu podstawowego. Krok ten zapewnia, że podprzestrzeń przenoszona do następnej iteracji jest możliwie jak najbardziej zwarta. Ocena odbywa się na podstawie amplitud zwróconych przez diagonalizację, gdyż reprezentują one ważność wkładu każdej konfiguracji do obliczonego stanu podstawowego.

Sprawdzenie zbieżności rozstrzyga, czy dalsze trenowanie mogłoby poprawić wyniki. Jeśli tak, wykonywany jest opcjonalny krok klasycznej ekspansji, parametry Circuit kwantowego są aktualizowane w celu dalszej minimalizacji obliczonej energii, a proces się powtarza. Krok klasycznej ekspansji generuje dodatkowe konfiguracje dla podprzestrzeni, uzupełniając konfiguracje próbkowane z urządzenia kwantowego. Najpierw identyfikuje konfigurację o największej amplitudzie w wynikach diagonalizacji, a następnie generuje nowe konfiguracje z pojedynczymi i podwójnymi wzbudzeniami z tej konfiguracji. Żądana liczba tych konfiguracji jest następnie dodawana do podprzestrzeni.

Gdy zostanie stwierdzona zbieżność iteracji, HI-VQE zwraca obliczony stan podstawowy (w postaci stanów w podprzestrzeni i ich amplitud w funkcji falowej stanu podstawowego), jego energię oraz miarę wariancji energii, która wskazuje, czy obliczony stan stanowi stan własny hamiltonianu układu.

Użytkownicy mogą decydować o używanym Circuit kwantowym i liczbie strzałów (shots) dla każdego Circuit kwantowego, a także kontrolować rozmiar podprzestrzeni lub włączyć klasyczne generowanie dodatkowych konfiguracji wspomagających konfiguracje generowane kwantowo. Dzięki temu użytkownicy mogą dostosować zachowanie HI-VQE do swoich docelowych zastosowań.

## Pierwsze kroki
Najpierw [poproś o dostęp do funkcji](https://forms.office.com/r/zN3hvMTqJ1).
Następnie uwierzytelnij się przy użyciu swojego [klucza API IBM Quantum&reg;](http://quantum.cloud.ibm.com/) i — zakładając, że masz już [zapisane konto](/guides/functions#install-qiskit-functions-catalog-client) w środowisku lokalnym — załaduj funkcję Qiskit w następujący sposób:

In [ ]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Dane wejściowe
W poniższej tabeli znajdziesz wszystkie parametry wejściowe przyjmowane przez funkcję. Kolejne sekcje [Opcje cząsteczki](#molecule-options) oraz [Opcje HI-VQE](#hi-vqe-options) omawiają te argumenty szczegółowo.

| Nazwa                  | Typ                                                            | Opis                                                                                                                                                                                                                                                                                                        | Wymagany | Domyślnie                                                                | Przykład                                                                                  |
|------------------------|----------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| geometry               | Union[List[List[Union[str, Tuple[float, float, float]]]], str] | Może być ciągiem znaków lub strukturalnymi listami zawierającymi pary atom–współrzędne. Jeśli podany jest jako ciąg, musi to być geometria cząsteczki w formacie Cartesian Coordinate Format. Jeśli podany jest jako lista, powinien być listą list zawierających ciąg atomu i krotkę współrzędnych. | Tak      | N/A                                                                      | `[['O', (0, 0, 0)], ['H', (0, 1, 0)], ['H', (0, 0, 1)]]` lub `"O 0 0 0; H 0 1 0; H 0 0 1"` |
| backend\_name          | str                                                            | Nazwa Backend, do którego kierowane jest zapytanie.                                                                                                                                                                                                                                                         | Tak      | N/A                                                                      | `ibm_fez`                                                                                 |
| max\_states            | int                                                            | Maksymalny wymiar podprzestrzeni przeznaczonej do diagonalizacji. Zostanie użyta mniejsza liczba stanów, jeśli podana wartość nie jest idealnym kwadratem.                                                                                                                                                                                                                                                    | Yes      | N/A                                                                      | `100`                                                                                     |
| max\_expansion\_states | int                                                            | Maksymalna liczba klasycznie generowanych stanów CI dołączanych w każdej iteracji.                                                                                                                                                                                                                     | Yes      | N/A                                                                      | `10`                                                                                      |
| molecule_options                | dict                                                           | Opcje związane z cząsteczką przekazywaną jako dane wejściowe do HI-VQE. Więcej informacji znajdziesz w sekcji [Opcje cząsteczki](#molecule-options).                                                                                                                                                                                                                                                | No       | See the [Molecule options](#molecule-options) section for details.                                 | `{"basis": "sto3g", "unit": "angstrom" }`                               |
| hivqe_options                | dict                                                           | Opcje sterujące zachowaniem algorytmu HI-VQE. Więcej informacji znajdziesz w sekcji [Opcje HI-VQE](#hi-vqe-options).                                                                                                                                                                                                                                                | No       | See the [HI-VQE options](#hi-vqe-options) section for details.                                 | `{"shots": 10_000, "max_iter": 10 }`                               |
### Opcje cząsteczki
Poniższa tabela opisuje wszystkie klucze i wartości, które można ustawić w słowniku `molecule_options`, wraz z ich typami danych i wartościami domyślnymi. Wszystkie klucze są opcjonalne.

| Key                               | Value type                          | Default Value                    | Valid range                                                                                                                                                    | Explanation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"charge"`                        | `int`                               | `0`                              | Various                                                                                                                                                        | Liczba całkowita określająca całkowity ładunek netto układu molekularnego. Wartość domyślna wynosi 0, ale może być dowolną liczbą całkowitą.                                                                                                                                                                                                                                                                                                                                                                                              |
| `"basis"`                         | `str`                               | `'sto-3g'`                       | Various                                                                                                                                                        | Ciąg znaków określający typ bazy; wartości te są przekazywane do pyscf. (np.: `"sto-3g"`, `"3-21g"`, `"6-31g"`, `"cc-pvdz"`)                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"active_orbitals"`               | `List[int]`                         | Every orbital index.             | The spatial orbital indices valid for the problem.                                                                                                             | Lista indeksów aktywnych orbitali w przedziale [0, n), gdzie n to liczba Qubitów użytych w zadaniu. Jeśli ten parametr jest podany, argument frozen_orbitals musi być również określony.                                                                                                                                                                                                                                                                                                                                            |
| `"frozen_orbitals"`               | `List[int]`                         | No indices.                      | The spatial orbital indices valid for the problem, excluding active orbitals.                                                                                  | Lista indeksów zamrożonych orbitali w tym samym zakresie co active_orbitals. Jeśli jest podany, active_orbitals musi być również określony. Pamiętaj, że zamrażać należy wyłącznie orbitale obsadzone, ponieważ liczba aktywnych elektronów zmniejsza się o 2 dla każdego zamrożonego orbitalu obsadzonego.                                                                                                                                                                                                                                        |
| `"orbital_coeffs"`                | `List[List[float]]`                 | Hartree-Fock molecular orbitals. | Various.                                                                                                                                                       | Współczynniki dla orbitali przestrzennych używanych przy obliczaniu całek odpychania elektronowego w układzie. Poprawnymi przykładami są orbitale molekularne Hartree-Focka, orbitale naturalne oraz orbitale AVAS.                                                                                                                                                                                                                                                                                                                   |
| `"symmetry"`                      | `Union[str, bool]`                  | `False`                          | `True` or `False`                                                                                                                                              | Służy do włączenia symetrii grupy punktowej w początkowych obliczeniach molekularnych, mających na celu konstrukcję bazy orbitalnej dopasowanej do symetrii. Te orbitale dopasowane do symetrii są używane jako funkcje bazowe w kolejnych obliczeniach SCF. Wartość domyślna to False; po ustawieniu na True symetria zostanie włączona, a arbitralne grupy punktowe będą wykrywane i stosowane automatycznie. Jeśli zostanie przypisana konkretna symetria, na przykład symmetry = "Dooh", błąd zostanie zgłoszony, jeśli geometria molekularna nie spełnia wymaganej symetrii. |
| `"symmetry_subgroup"`             | `Optional[str]`                     | `None`                           | See [pyscf documentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Może być używany do wygenerowania podgrupy wykrytej symetrii. Nie ma żadnego efektu, gdy symetria jest określona za pomocą argumentu kluczowego symmetry.                                                                                                                                                                                                                                                                                                                                                                         |
| `"unit"`                          | `str`                               | `"angstrom"`                     | See [pyscf documentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Określa jednostkę miary używaną do współrzędnych atomowych i odległości. Domyślnie stosowane są angstromy.                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"nucmod"`                        | `Optional[Union[dict, str]]`        | `None`                           | See [pyscf documentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Określa model jądrowy, który ma być używany. Domyślnie stosowany jest punktowy model jądrowy; inne wartości włączają gaussowski model jądrowy. Jeśli podana zostanie funkcja, zostanie ona użyta wraz z gaussowskim modelem jądrowym do wygenerowania wartości rozkładu ładunku jądrowego 'zeta'.                                                                                                                                                                                                                                                   |
| `"pseudo"`                        | `Optional[Union[dict, str]]`        | `None`                           | See [pyscf documentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Określa pseudopotencjał dla atomów w cząsteczce. Domyślnie ma wartość None, co oznacza, że żadne pseudopotencjały nie są stosowane, a wszystkie elektrony są jawnie uwzględniane w obliczeniach.                                                                                                                                                                                                                                                                                                                                                |
| `"cart"`                          | `bool`                              | `False`                          | See [pyscf documentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Określa, czy w obliczeniach mają być używane kartezjańskie GTO jako funkcje bazowe momentu pędu. Domyślna wartość False spowoduje użycie sferycznych GTO.                                                                                                                                                                                                                                                                                                                                                               |
| `"magmom"`                        | `Optional[List[Union[int, float]]]` | `None`                           | See [pyscf documentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Ustawia kolinearny spinowy moment magnetyczny każdego atomu. Domyślnie ma wartość None i każdy atom jest inicjalizowany ze spinem równym zero.                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_aolabels"`                 | `Optional[List[str]]`               | `None`                           | i.e. ["H 1s", "O 2p"] for H2O                                                                                                                                                             | Definiuje orbitale atomowe (AO) do uwzględnienia w schemacie AVAS. Zobacz [dokumentację AVAS](https://arxiv.org/abs/1701.07862).                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_threshold"`                | `float`                             | `0.2`                            |  Between 0.0 and 2.0                                                                                                                                                      |  Określa wartość graniczną używaną do decydowania, które orbitale atomowe (AO) są zachowywane w aktywnej przestrzeni.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"noons_level"`                   | `Optional[str]`                     | `None`                           | `"mp2"` or `"ccsd"`                                                                                                                                            | Definiuje podejście teoretyczne do przygotowania orbitali naturalnych i selekcji aktywnych orbitali na podstawie schematu Liczb Obsadzenia Orbitali Naturalnych (NOONs). Zobacz [dokumentację NOONs](https://doi.org/10.1063/5.0042147). Aby kontrolować liczbę orbitali (i liczbę Qubitów), należy podać zarówno indeksy aktywnych, jak i zamrożonych orbitali.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
### Opcje HI-VQE
Poniższa tabela opisuje wszystkie klucze i wartości, które można ustawić w słowniku `hivqe_options`, wraz z ich typami danych i wartościami domyślnymi. Wszystkie klucze są opcjonalne.

| Key                               | Value type                          | Default Value                    | Valid range                                                                                                                                                    | Explanation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"shots"`                         | `int`                               | `1_000`                          | Between 1 and 10 000.                                                                                                                                          | Liczba pomiarów (shots) do wykonania na urządzeniu kwantowym na iterację.                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| `"max_iter"`                      | `int`                               | `25`                             | Between 1 and 50.                                                                                                                                              | Maksymalna liczba iteracji do przeprowadzenia w celu optymalizacji ansatzu. Algorytm może użyć mniejszej liczby iteracji, jeśli zbieżność zostanie osiągnięta wcześniej.                                                                                                                                                                                                                                                                                                                                                                 |
| `"initial_basis_states"`          | `List[str]`                         | The Hartree-Fock state.          | Binary strings with the number of bits matching the required number of qubits for the problem.                                                                 | Może być używany do ponownego uruchomienia algorytmu ze stanami klasycznymi z poprzedniego wyniki.                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz"`                        | `str`                               | `"epa"`                          | `"epa"`, `"hea"`, or `"lucj"`                                                                                                                                  | Określa kwantowy ansatz do optymalizacji w celu generowania nowych stanów. `"epa"` wybiera ansatz zachowujący wzbudzenia. `"hea"` wybiera ansatz wydajny sprzętowo. `"lucj"` wybiera lokalny unitarny ansatz cluster Jastrow.                                                                                                                                                                                                                                                                                                       |
| `"convergence_count"`             | `int`                               | `3`                              | At least 2.                                                                                                                                                    | Liczba iteracji bez znaczącej zmiany obliczonej energii, jaka musi upłynąć, zanim algorytm zostanie uznany za zbieżny.                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"convergence_abstol"`            | `float`                             | `1e-4`                           | More than 0 and at most 1.                                                                                                                                     | Wartość zmiany obliczonej energii uznawana za znaczącą na potrzeby sprawdzania zbieżności.                                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"reset_convergence_count"`       | `bool`                              | `True`                           | `True` or `False`                                                                                                                                              | Jeśli `True`, wymagana liczba iteracji `convergence_count` musi wystąpić bez przerywającej znaczącej zmiany, aby kwalifikować się jako zbieżność. Jeśli `False`, algorytm zatrzyma się po `convergence_count` iteracjach, jeśli w jakiejkolwiek iteracji podczas procesu optymalizacji wystąpiły nieznaczące zmiany.                                                                                                                                                                 |
| `"configuration_recovery"`        | `bool`                              | `True`                           | `True` or `False`.                                                                                                                                             | Określa, czy ma być używane odtwarzanie konfiguracji z pakietu `qiskit-addon-sqd`. Jeśli True, nieprawidłowe stany próbkowane z urządzenia kwantowego są poprawiane klasycznie. Jeśli False, są odrzucane.                                                                                                                                                                                                                                                                                                                      |
| `"ansatz_entanglement"`           | `str`                               | `"circular"`                     | Any one of `"linear"`, `"reverse_linear"`, `"pairwise"`, `"circular"`, `"full"`, or `"sca"`. If using the `"lucj"` ansatz, `"lucj_default"` is also an option. | Określa schemat splątania używany wewnątrz Circuit kwantowego, zgodnie ze wspólnymi konwencjami Qiskit i [konwencjami ffsim dla ansatzu LUCJ](https://qiskit-community.github.io/ffsim/how-to-guides/simulate-lucj.html).                                                                                                                                                       |
| `"ansatz_reps"`                   | `int`                               | `2`                              | Greater than 0.                                                                                                                                                | Liczba powtórzeń każdej warstwy w Circuit kwantowym.                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"amplitude_screening_tolerance"` | `Union[float,int]`                  | `0`                              | At least 0, and less than 1.                                                                                                                                   | Tolerancja decydująca o tym, które stany powinny być odsiewane z podprzestrzeni po diagonalizacji. Określa próg uwzględniania stanów podprzestrzeni na podstawie ich obliczonych amplitud.                                                                                                                                                                                                                                                                                                                                  |
| `"overlap_screening_tolerance"`   | `float`                             | `1e-2`                           | Between `1e-4` and `1e-1`, inclusive.                                                                                                                          | Tolerancja służąca do przewidywania, które stany powinny być odsiewane z podprzestrzeni przed diagonalizacją. Kontroluje dokładność przewidywanych amplitud dla każdego stanu — mniejsza wartość skutkuje dokładniejszymi przewidywaniami.                                                                                                                                                                                                                                                                                            |
## Dane wyjściowe
Funkcja zwraca słownik z czterema kluczami i wartościami. Klucze i wartości są opisane w poniższej tabeli:

| Key             | Value Type    | Explanation                                                                                                               |
|:----------------|---------------|---------------------------------------------------------------------------------------------------------------------------|
| `"energy"`      | `float`       | Przybliżona energia stanu podstawowego cząsteczki.                                                                      |
| `"states"`      | `List[str]`   | Wybrane determinanty tworzące podprzestrzeń użytą do obliczenia energii. Są w naprzemiennym formacie alfa-beta. |
| `"eigenvector"` | `List[float]` | Wektor własny odpowiadający stanowi podstawowemu podprzestrzeni złożonej ze `"states"`.                                 |
| `"energy_variance"` | `float` | Wariancja energii stanu podstawowego podprzestrzeni złożonej ze `"states"`, stanowiąca wskazówkę dotyczącą jakości rozwiązania. Wartość ta jest nieujemna, a niższa wartość oznacza, że stan podstawowy podprzestrzeni jest bliższy stanowi własnemu Hamiltonianu układu. |
| `"energy_history"` | `List[float]` | Energie obliczane w każdej iteracji podczas hybrydowego procesu optymalizacji, w kolejności, w jakiej zostały obliczone. Dwie energie są obliczane na iterację jako część procesu optymalizacji SPSA. |
## Przykład
Pierwszy przykład pokazuje, jak obliczyć energię stanu podstawowego cząsteczki NH3 przy użyciu algorytmu HI-VQE.
#### Zdefiniuj geometrię molekularną i opcje
Geometria molekularna NH3 jest podana we współrzędnych kartezjańskich oddzielonych znakiem ";" dla każdego atomu.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Dodatkowe opcje można zdefiniować i przekazać dla układu molekularnego w następującym formacie słownika.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Uruchom funkcję, podając geometrię i opcje jako dane wejściowe.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

Dobrym pomysłem jest wydrukowanie identyfikatora zadania Function, aby móc go podać w zgłoszeniu do pomocy technicznej, jeśli coś pójdzie nie tak.

In [6]:
print("Job ID:", job.job_id)

Job ID: 128ee399-7cfc-4be2-91e9-c4abd22b97c7


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


Ten przykład wykorzystuje 16 Qubitów z 8 orbitalami bazy sto3g dla cząsteczki NH3.
Sprawdź [status](/guides/functions#check-job-status) zadania Qiskit Function lub pobierz [wyniki](/guides/functions#retrieve-results) w następujący sposób:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824200343205695, 0.009520846167419264, 6.365368845740147e-08, 3.6832123006425615e-07, 0.0012869941182066654, 2.3403891050875465e-05, ...], 'energy': -55.521146537970466, 'energy_history': [-55.52091922342852, -55.52113695367561, -55.521146537970466, -55.52114653864798, -55.521146537970466, -55.521146537970466, ...], 'energy_variance': 9.788555455342562e-12, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [10]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.0014967336596782843 mHa
Sampled Number of States: 1936


Po zakończeniu zadania wyniki można uzyskać za pomocą instancji `result()`.

In [11]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [12]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

Aby uzyskać dostęp do energii stanu podstawowego, użyj klucza `"energy"`. Klucz `"eigenvector"` dostarcza współczynniki CI wraz z odpowiadającą im notacją bitstring konfiguracji elektronowej, przechowywaną pod kluczem `"states"` wyników.

In [13]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: {"message": "An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance.", "status": "failure"}

In [14]:
job.status()

'ERROR'

Wynik:

|Exact Energy - HI-VQE Energy|: 0.077237504 mHa
Sampled Number of States: 1936
## Wydajność
W tej sekcji przedstawiono wyniki wzorcowych obliczeń HI-VQE: przypadek 24-Qubitowy dla Li2S, przypadek 40-Qubitowy dla cząsteczki N2 oraz przypadek 44-Qubitowy dla układu FeP-NO.
#### Krzywa potencjalnej powierzchni energii dysocjacji dla cząsteczki Li2S z 24 Qubitami
Krzywa PES jest pokazana wraz z referencją FCI i początkowym przybliżeniem z RHF, a także błędem energii względem referencji FCI.

![Obraz pokazujący, że HI-VQE daje rozwiązania mieszczące się w dokładności chemicznej klasycznej referencyjnej krzywej PES dla układu Li2S.](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

Obliczenia zostały przeprowadzone dla następujących geometrii i opcji.